# Simulación del control del volumen de líquido de un depósito

Antes de empezar, importamos las librerías necesarias:

In [ ]:
import threading
import time
import matplotlib.pyplot as plt

Necesitaremos declarar algunas variables globales.

La primera será una variable de tipo Lock para implementar el semáforo (mutex en C):

In [ ]:
lock = threading.Lock()

Necesitaremos almacenar los datos de la simulación (volumen de líquido del depçosito en cada instante de control), así, crearemos una tupla para ello:

In [ ]:
datos = []

Por último iniciaremos el tiempo y el volumen a sus valores iniciales:

In [ ]:
tiempo = 0
volumen = 0

Leemos los datos de entrada al simulador:

In [ ]:
tiempoej = 20
caudal = 5
vaciado = 10
umbral = 13
peligro = 21

# DESCOMENTAR PARA INTRODUCIR LOS DATOS DE L SIMULACIÓN POR TECLADO

# tiempoej = int(input("Introduce el tiempo de simulación (seg.): "))
# caudal = int(input("Introduce el valor del llenado (l./seg.): "))
# vaciado = int(input("Introduce el valor del vaciado (l.): "))
# umbral = int(input("Introduce el valor de control(l): "))
# peligro = int(input ("Introduce el valor de activación de la señar de alarma: "))

# Para asegurarnos de que los datos introducidos son realmente números enteros
# debemos usar una estructura try/except en cada lectura
# Ejemplo
# 
# try:
#   numero = int(input("Introduce un número entero: "))
#   print(f"El número ingresado es: {numero}")
# except ValueError:
#    print("¡Error! Debes introducir un número entero válido.")
 

Nos aseguramos de que 'lock' existe y es un objeto threadding.lock válido

In [ ]:
# Asegurarnos de que 'lock' existe y es un threading.Lock válido antes de usarlo
try:
    _tmp = lock
    if not (hasattr(lock, '__enter__') and hasattr(lock, '__exit__')):
        lock = threading.Lock()
except NameError:
    lock = threading.Lock()

Para generar una vista más cómoda, definiremos las funciones que ejecuta cada thread del fichero depo.py

In [ ]:
#from depo import controldep, alarma, llenado, reloj (no funciona)???

def controldep():
    global tiempo, volumen, umbral, vaciado, datos, lock
    while (tiempo < tiempoej):
        with lock:
            if (volumen > umbral):
                volumen = volumen - vaciado # type: ignore
                #print(f"Control: he reducido el volumen a {volumen}")
        time.sleep(2)
        
def alarma():
    global tiempo, volumen, peligro, lock
    while (tiempo < tiempoej):
        with lock:
            if (volumen > peligro):
                continue
                #print(f"Alarma: cuidado, el volumen es: {volumen}")
        time.sleep(1)
      
def llenado():
    global tiempo, tiempoej, volumen, caudal, lock
    while (tiempo < tiempoej):
        with lock:
            volumen = volumen + caudal
            datos.append(volumen)
            #print(f"Llenado : volumen = {volumen}")
        time.sleep(1)
        
def reloj():
    global tiempo, inicio, tiempoej
    while (tiempo<tiempoej):
        fin = time.time()
        tiempo = fin - inicio
        #print(tiempo)

Definimos cada uno de los threads que van a ejecutar las distintas funciones:

In [ ]:
th_cont = threading.Thread(target=controldep)
th_alar = threading.Thread(target=alarma)
th_llena = threading.Thread(target=llenado)
th_reloj = threading.Thread(target=reloj)

Iniciamos el tiempo de simulación:

In [ ]:
inicio = time.time()

Lanzamos los threads

In [ ]:
th_reloj.start()
th_llena.start()
th_cont.start()
th_alar.start()

Desde la celda anterior, el simulador se está ejecutando. Cuando termine, es necesario esperar la terminación de los distintos threads:

In [ ]:
th_cont.join()
th_alar.join()
th_llena.join()
th_reloj.join()

Calculamos (como curiosidad) el tiempo total de ejcución:

In [ ]:
final = time.time()
tejec = final - inicio
print (f"Tiempo de ejecución: {tejec}")

Ahora, representamos gráficamente la salida de la simulación:

In [ ]:
# Representar los datos
longitud_datos = len(datos)
ejex = []
for i in range(0, longitud_datos):
    ejex.append(i)

# print(len(datos))
# print(len(ejex))

plt.figure(figsize=(12, 8)) # Opcional: Define el tamaño de la figura
plt.plot(ejex, datos, label='Volumen de líquido', color='blue', linestyle='solid') # Dibuja la línea

# 3. Personalizar (opcional)
plt.title('Control depósito')
plt.xlabel('Muestras (T = 0.7 seg.)')
plt.ylabel('Volumen (l.)')
plt.legend() # Muestra la leyenda
plt.grid(True) # Añade una cuadrícula

# 4. Mostrar la gráfica
plt.show()        
